# Chaos testing

Chaos testing settings. 

In [ ]:
# Notebook-friendly chaos test (fixture-like via contextmanager), complete segment.

import os
from contextlib import contextmanager
from time import perf_counter

import httpx
import openai
import requests

# --- Config ---
API_KEY = "sk-local-dev-b835e4582d0619bafcdc6ee5bdeab433"

# clean and chaos ingress
BASE_URL_CLEAN = "https://caddy:4000"
BASE_URL_CHAOS = "https://caddy:4001"

# local CA from your compose cert setup
CA_CERT_PATH = "/certs/.caroot/rootCA.pem"
VERIFY_CONFIG = CA_CERT_PATH if os.path.exists(CA_CERT_PATH) else True

# toxiproxy config
TOXI_URL = "http://toxiproxy:8474"
PROXY_NAME = "edge_chaos"
TOXIC_NAME = "nb_edge_latency"

# llm request config
MODEL = "ollama_chat/llama3.2:3b"
PROMPT = "Sag kurz Hallo auf Deutsch."


def make_client(base_url: str) -> openai.OpenAI:
    http_client = httpx.Client(verify=VERIFY_CONFIG, timeout=60.0)
    return openai.OpenAI(
        api_key=API_KEY,
        base_url=base_url,
        http_client=http_client,
    )


def timed_chat(base_url: str, model: str = MODEL, prompt: str = PROMPT):
    client = make_client(base_url)
    start = perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    elapsed = perf_counter() - start
    text = response.choices[0].message.content or ""
    return elapsed, text


def remove_toxic_if_present():
    # Ignore errors if toxic does not exist yet
    try:
        requests.delete(
            f"{TOXI_URL}/proxies/{PROXY_NAME}/toxics/{TOXIC_NAME}",
            timeout=5,
        )
    except Exception:
        pass


@contextmanager
def edge_latency_fixture(latency_ms=700, jitter_ms=100):
    # setup
    r = requests.post(
        f"{TOXI_URL}/proxies/{PROXY_NAME}/toxics",
        json={
            "name": TOXIC_NAME,
            "type": "latency",
            "stream": "downstream",  # litellm -> client
            "toxicity": 1.0,
            "attributes": {"latency": latency_ms, "jitter": jitter_ms},
        },
        timeout=5,
    )
    r.raise_for_status()
    try:
        yield
    finally:
        # teardown
        remove_toxic_if_present()


time_chat = timed_chat(BASE_URL_CLEAN) # Load model, startup delay!
print(f"[pre-flight to start model] duration={time_chat[0]:.2f}s\n{time_chat[1]}\n")
# --- Run experiment ---
remove_toxic_if_present()  # clean start

clean_s, clean_text = timed_chat(BASE_URL_CHAOS)  # same ingress, no toxic
print(f"[baseline] duration={clean_s:.2f}s\n{clean_text}\n")

with edge_latency_fixture(latency_ms=900, jitter_ms=100):
    chaos_s, chaos_text = timed_chat(BASE_URL_CHAOS)
    print(f"[chaos] duration={chaos_s:.2f}s\n{chaos_text}\n")



# notebook assertions
assert chaos_text is not None
assert chaos_s > clean_s + 0.4, f"Expected clear delay. clean={clean_s:.2f}s chaos={chaos_s:.2f}s"

cleanup_s, cleanup_text = timed_chat(BASE_URL_CHAOS) # Ensure we have no delay
print(f"[post-flight to verify clear line] duration={cleanup_s:.2f}s\n{cleanup_text}\n")
assert cleanup_text is not None
assert cleanup_s < chaos_s

print("OK: edge latency toxic had measurable effect and was cleaned up.")

[pre-flight to start model] duration=2.74s
Hallo! Wie kann ich Ihnen helfen?

[baseline] duration=0.92s
Hallo! Wie kann ich dir helfen?

[chaos] duration=1.95s
Hallo! Wie kann ich Ihnen helfen?

[post-flight to verify clear line] duration=2.74s
Hallo! Wie kann ich Ihnen helfen?

OK: edge latency toxic had measurable effect and was cleaned up.
